# PASAR DE UN DATASET HORARIO A DIARIO HACIENDO LA MEDIA DIARIA

In [4]:
import pandas as pd

input_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_semicompletos/df__Algar_Palancia_completo.csv"         # <- pon aquí tu CSV horario
output_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_media_diaria/df__Algar_Palancia_diario_vfq.csv" # <- nombre del CSV de salida

# leer y parsear datetime
df = pd.read_csv(input_path, parse_dates=["datetime"])

# asegurarse de que datetime sea índice
df = df.set_index("datetime")

# columnas objetivo (se ignorarán si no existen)
cols = ["NO", "NO2", "NOx", "O3"]
available_cols = [c for c in cols if c in df.columns]
if not available_cols:
    raise ValueError("No se encontraron columnas NO, NO2, NOx ni O3 en el CSV de entrada.")

# resample diario y calcular la media (ignora NaNs por defecto)
daily = df[available_cols].resample("D").mean()

# convertir índice a columna 'datetime' con frecuencia diaria (YYYY-MM-DD)
daily = daily.reset_index()
daily["datetime"] = daily["datetime"].dt.strftime("%Y-%m-%d")

# guardar CSV resultante
daily.to_csv(output_path, index=False)
print(f"CSV de medias diarias guardado en: {output_path}")


CSV de medias diarias guardado en: /Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_media_diaria/df__Algar_Palancia_diario_vfq.csv


# CAMBIAR NOMBRE Y UNIDADES DATOS AEMET

In [7]:
import pandas as pd

input_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos-AEMET/Datosmeteorologicossegorbe.csv"         # <- pon aquí tu CSV horario
output_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos-AEMET/Datosmeteorologicossegorbe_limpios.csv" # <- nombre del CSV de salida


# 1) leer CSV
df = pd.read_csv(input_path)

# 2) convertir 'dir' a numérico (si hay valores no numéricos quedarán NaN)
df['dir'] = pd.to_numeric(df['dir'], errors='coerce')

# 3) pasar dir 0-100 -> 0-360
#    grados = dir * 360 / 100
#    si prefieres que 100 -> 0 (equivalente), usa % 360; aquí dejamos el valor directo
df['Direc.'] = (df['dir'] * 360.0 / 100.0)

#  opcional: si quieres normalizar 360 -> 0 (valores en [0,360) en lugar de [0,360])
# df['Direc.'] = df['Direc.'] % 360

# 4) renombrar las otras columnas
rename_map = {
    "fecha": "datetime",
    "tmed": "Temp.",
    "velmedia": "Veloc."
}
df = df.rename(columns=rename_map)

# 5) eliminar la columna original 'dir' (ya tenemos 'Direc.')
df = df.drop(columns=['dir', 'indicativo', 'nombre', 'provincia', 'altitud', 'prec', ], errors='ignore')

# 6) asegurar formato de datetime (entrada en formato 'YYYY-MM-DD')
#    si hay filas con fecha inválida quedarán NaT -> luego se formatean como NaN en CSV
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce').dt.strftime("%Y-%m-%d")

# 7) reordenar columnas (opcional) para tener un CSV limpio
cols_desired = ['datetime', 'Temp.', 'Direc.', 'Veloc.']
# mantener cualquier otra columna adicional al final
other_cols = [c for c in df.columns if c not in cols_desired]
df = df[[c for c in cols_desired if c in df.columns]]

# 8) guardar el CSV resultante
df.to_csv(output_path, index=False)
print(f"Fichero guardado en: {output_path}")



Fichero guardado en: /Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos-AEMET/Datosmeteorologicossegorbe_limpios.csv


# COMBINA LAS VARIABLES MET (AEMET) CON LAS F-Q (GVA) EN UN UNICO CSV

In [9]:
import pandas as pd

# --- Ajusta estas rutas ---
csv_aires_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_media_diaria/df__Algar_Palancia_diario_vfq.csv"     # CSV que contiene NO, NO2, NOx, O3
csv_meteo_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos-AEMET/Datosmeteorologicossegorbe_limpios.csv"     # CSV que contiene Temp., Veloc., Direc.
output_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos-SEGORBE/Datosmet+F-Qsegorbe_SIN_R.SOL.csv"     # CSV resultante
# ---------------------------

# Leer CSVs (sin parseo forzado aún)
df_aires = pd.read_csv(csv_aires_path)
df_meteo = pd.read_csv(csv_meteo_path)

# Función auxiliar para detectar y convertir la columna de fecha a 'datetime'
def ensure_datetime_col(df):
    # posibles nombres de la columna de fecha
    candidates = ["datetime", "fecha", "DateTime", "date", "Fecha"]
    found = None
    for c in candidates:
        if c in df.columns:
            found = c
            break
    if found is None:
        raise KeyError(f"No se encontró columna de fecha. Busqué: {candidates}")
    # convertir a datetime y renombrar a 'datetime'
    df = df.copy()
    df["datetime"] = pd.to_datetime(df[found], errors="coerce")
    # eliminar la columna original si se llamaba distinto
    if found != "datetime":
        df = df.drop(columns=[found])
    return df

# Aplicar conversión
df_aires = ensure_datetime_col(df_aires)
df_meteo = ensure_datetime_col(df_meteo)

# Comprobar columnas objetivo y construir lista de columnas a seleccionar
cols_aires = ["NO", "NO2", "NOx", "O3"]
cols_meteo = ["Temp.", "Veloc.", "Direc."]

avail_aires = [c for c in cols_aires if c in df_aires.columns]
avail_meteo = [c for c in cols_meteo if c in df_meteo.columns]

if not avail_aires:
    raise ValueError(f"No se encontraron columnas de contaminantes en {csv_aires_path}. Busqué: {cols_aires}")
if not avail_meteo:
    raise ValueError(f"No se encontraron columnas meteorológicas en {csv_meteo_path}. Busqué: {cols_meteo}")

# Seleccionar columnas necesarias (incluimos 'datetime' en ambos para merge)
df_aires_sel = df_aires[["datetime"] + avail_aires].copy()
df_meteo_sel = df_meteo[["datetime"] + avail_meteo].copy()

# Opcional: eliminar filas con datetime inválido (NaT)
df_aires_sel = df_aires_sel.dropna(subset=["datetime"])
df_meteo_sel = df_meteo_sel.dropna(subset=["datetime"])

# Asegurar que no haya duplicados de datetime (si los hay, los agrupamos tomando la media)
if df_aires_sel["datetime"].duplicated().any():
    df_aires_sel = df_aires_sel.groupby("datetime", as_index=False)[avail_aires].mean()
if df_meteo_sel["datetime"].duplicated().any():
    df_meteo_sel = df_meteo_sel.groupby("datetime", as_index=False)[avail_meteo].mean()

# Hacer el merge por 'datetime' (por defecto inner: solo comunes)
merged = pd.merge(df_aires_sel, df_meteo_sel, on="datetime", how="inner", validate="one_to_one")

# Si quieres un merge distinto (por ejemplo mantener todos los tiempos), cambia how="inner" por "outer" o "left"
# merged = pd.merge(df_aires_sel, df_meteo_sel, on="datetime", how="outer", validate="one_to_one")

# Opcional: ordenar por datetime
merged = merged.sort_values("datetime").reset_index(drop=True)

# Formatear datetime en el CSV de salida (ej: YYYY-MM-DD o con hora si la tienen)
# Si quieres que conserve hora: comentarlo para dejar datetime como datetime64 en el DataFrame pero al guardar se convierte a string.
merged["datetime"] = merged["datetime"].dt.strftime("%Y-%m-%d %H:%M:%S")

# Guardar CSV final
merged.to_csv(output_path, index=False)
print(f"CSV combinado guardado en: {output_path}")
print(f"Columnas incluidas: {list(merged.columns)}")
print(f"Nº filas resultantes: {len(merged)}")


CSV combinado guardado en: /Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos-SEGORBE/Datosmet+F-Qsegorbe_SIN_R.SOL.csv
Columnas incluidas: ['datetime', 'NO', 'NO2', 'NOx', 'O3', 'Temp.', 'Veloc.', 'Direc.']
Nº filas resultantes: 5006
